# Phase 5 — Kaggle Submission

Loads the trained ensemble, generates predictions via ensemble + MC Dropout averaging, and writes `submission.csv`.

## 1. Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Shared config across Phase 3/4/5
INPUT_DIR    = 'data/'
MODEL_DIR    = 'saved_models_v2/'
ENSEMBLE_SIZE = 5
HIDDEN_DIM   = 256
ESM_DIM      = 1280
NODE_FEAT    = 29
EDGE_FEAT    = 7
N_GIN_LAYERS = 4
DROPOUT      = 0.2
BATCH_SIZE   = 128

In [ ]:
class DTA_Model(nn.Module):
    """
    Drug-Target Affinity predictor.
    
    Improvements over the previous GCN approach:
    - GINEConv: uses bond features (edge_attr), more expressive than GCN
    - BatchNorm: stabilizes training, prevents the wild RMSE oscillations
    - Morgan fingerprint branch: uses the 2048-bit FPs computed in Phase 2
    - Dropout throughout: enables MC Dropout for uncertainty in Phase 4
    """

    def __init__(self, node_feat=29, edge_feat=7, esm_dim=1280,
                 hidden_dim=256, n_layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # --- Drug graph encoder (GINEConv) ---
        self.edge_proj = nn.Linear(edge_feat, hidden_dim)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(n_layers):
            in_dim = node_feat if i == 0 else hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # --- Drug fingerprint encoder ---
        self.fp_proj = nn.Sequential(
            nn.Linear(2048, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # --- Protein encoder ---
        self.prot_proj = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # --- Prediction head ---
        # graph(256) + fp(256) + protein(256) = 768
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def get_compound_embedding(self, data):
        """Extract the compound graph embedding (used by Phase 4 for OOD detection)."""
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        edge_emb = self.edge_proj(edge_attr)
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_emb)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return global_mean_pool(x, batch)

    def forward(self, data):
        # 1. Drug graph
        drug_graph = self.get_compound_embedding(data)  # (B, hidden)
        # 2. Drug fingerprint
        drug_fp = self.fp_proj(data.fp.squeeze(1))      # (B, hidden)
        # 3. Protein
        prot = self.prot_proj(data.target_emb.squeeze(1))  # (B, hidden)
        # 4. Predict
        combined = torch.cat([drug_graph, drug_fp, prot], dim=-1)
        return self.head(combined).squeeze(-1)  # (B,)

## 2. Load Models & Data

In [ ]:
print('Loading test data...')
test_data = torch.load(os.path.join(INPUT_DIR, 'test_data.pt'))
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
test_ids = [d.sample_id for d in test_data]
print(f'Test samples: {len(test_data)}')

print(f'Loading {ENSEMBLE_SIZE} models...')
models = []
for i in range(1, ENSEMBLE_SIZE + 1):
    m = DTA_Model(node_feat=NODE_FEAT, edge_feat=EDGE_FEAT, esm_dim=ESM_DIM,
                  hidden_dim=HIDDEN_DIM, n_layers=N_GIN_LAYERS, dropout=DROPOUT).to(device)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'model_{i}.pt'), map_location=device))
    m.eval()
    models.append(m)
print('All models loaded.')

## 3. Ensemble + MC Dropout Inference

5 models × 10 MC passes = 50 forward passes. Averaging all of them gives much more stable predictions than the previous approach (3 models × 1 pass = 3). More stable predictions → lower RMSE → better Kaggle score.

In [ ]:
def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

N_MC_PASSES = 10
total_passes = ENSEMBLE_SIZE * N_MC_PASSES
print(f'Running {ENSEMBLE_SIZE} models × {N_MC_PASSES} MC passes = {total_passes} total\n')

all_forward_passes = []

for idx, model in enumerate(models):
    model.eval()
    enable_mc_dropout(model)
    
    for mc in range(N_MC_PASSES):
        pass_preds = []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                pass_preds.extend(model(batch).cpu().numpy())
        all_forward_passes.append(pass_preds)
    print(f'  Model {idx+1}/{ENSEMBLE_SIZE} done')

all_forward_passes = np.array(all_forward_passes)  # (50, 23191)
final_preds = np.mean(all_forward_passes, axis=0)
pred_std = np.std(all_forward_passes, axis=0)

print(f'\nPrediction range: [{final_preds.min():.3f}, {final_preds.max():.3f}]')
print(f'Mean uncertainty:  {pred_std.mean():.4f}')

## 4. Build & Save Submissions

In [ ]:
# Variant 1: Ensemble + MC Dropout mean (recommended)
sub_mc = pd.DataFrame({'id': test_ids, 'affinity': final_preds})

# Variant 2: Clipped to reasonable pKi range [3, 12]
sub_clip = pd.DataFrame({'id': test_ids, 'affinity': np.clip(final_preds, 3.0, 12.0)})

# Variant 3: Deterministic ensemble only (no MC dropout)
det_preds = []
for m in models:
    m.eval()  # fully deterministic (dropout off)
    batch_p = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            batch_p.extend(m(batch).cpu().numpy())
    det_preds.append(batch_p)
det_mean = np.mean(det_preds, axis=0)
sub_det = pd.DataFrame({'id': test_ids, 'affinity': det_mean})

In [ ]:
# Cross-check against sample_submission.csv if available
for path in ['google_drive/sample_submission.csv', 'sample_submission.csv', 'data/sample_submission.csv']:
    if os.path.exists(path):
        sample = pd.read_csv(path)
        expected = set(sample['id'])
        ours = set(sub_mc['id'])
        if expected == ours:
            print(f'ID match with {path}: PERFECT ✓ ({len(expected)} IDs)')
            # Reorder to match
            sub_mc = sample[['id']].merge(sub_mc, on='id', how='left')
            sub_clip = sample[['id']].merge(sub_clip, on='id', how='left')
            sub_det = sample[['id']].merge(sub_det, on='id', how='left')
        else:
            missing = expected - ours
            extra = ours - expected
            if missing: print(f'⚠️ Missing {len(missing)} IDs')
            if extra: print(f'⚠️ {len(extra)} extra IDs')
        break
else:
    print('sample_submission.csv not found — skipping cross-check')

In [ ]:
sub_mc.to_csv('submission.csv', index=False)
sub_clip.to_csv('submission_clipped.csv', index=False)
sub_det.to_csv('submission_deterministic.csv', index=False)

print('Saved 3 submission variants:')
print(f'  1. submission.csv              (MC+Ensemble, recommended)')
print(f'  2. submission_clipped.csv       (clipped to [3, 12])')
print(f'  3. submission_deterministic.csv  (ensemble only, no MC)')

## 5. Sanity Checks

In [ ]:
sub = pd.read_csv('submission.csv')
checks = [
    ('Columns = [id, affinity]', list(sub.columns) == ['id', 'affinity']),
    (f'Rows = {len(test_data)}', len(sub) == len(test_data)),
    ('No nulls', not sub['affinity'].isna().any()),
    ('No duplicate IDs', sub['id'].nunique() == len(sub)),
    ('Range reasonable', sub['affinity'].min() > 0 and sub['affinity'].max() < 15),
]
all_ok = True
for label, ok in checks:
    print(f'  [{"✓" if ok else "✗"}] {label}')
    all_ok = all_ok and ok

print(f'\nRange: [{sub["affinity"].min():.3f}, {sub["affinity"].max():.3f}]')
print(f'Mean: {sub["affinity"].mean():.3f}')
print(f'\n{"🚀 Ready to upload!" if all_ok else "⚠️ Fix issues first."}')

## Upload Instructions

1. Go to https://www.kaggle.com/competitions/star-ai-data-competition-track-2/submit
2. Upload `submission.csv`
3. Check RMSE on public leaderboard
4. Try the other two variants
5. **Select your best as final** before the deadline

Remember: final ranking uses the private leaderboard (60% of test data), so don't overfit to the public score.